# 1.10 The No Free Lunch theorem

Averaged over all possible problems, every learning algorithm is exactly as good as random guessing: no universal learner exists.

No Free Lunch is an enumeration theorem over uniformly likely target functions. It does not say learning is hopeless; it says advantages require inductive bias and disappear under the uniform ensemble of all labelings.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(16)


def theory_ladder(topic):
    """Return compact D1--D5 inputs for F16 theory notebooks."""
    if topic == "srm":
        return [
            {"name": "D1 lesson four-rung ladder", "emp": np.array([0.30, 0.15, 0.07, 0.05]), "H": np.array([2, 16, 256, 65536]), "m": 200, "delta": 0.05},
            {"name": "D2 six nested polynomial rungs", "emp": np.array([0.34, 0.21, 0.13, 0.09, 0.075, 0.07]), "H": np.array([2, 8, 32, 128, 512, 2048]), "m": 200, "delta": 0.05},
            {"name": "D3 larger sample shifts upward", "emp": np.array([0.30, 0.15, 0.07, 0.04, 0.035]), "H": np.array([2, 16, 256, 65536, 1048576]), "m": 2000, "delta": 0.05},
            {"name": "D4 noisy empirical errors", "emp": np.array([0.31, 0.19, 0.12, 0.115, 0.13]), "H": np.array([2, 16, 256, 4096, 65536]), "m": 300, "delta": 0.05},
            {"name": "D5 bad ordering stress case", "emp": np.array([0.28, 0.10, 0.03, 0.02, 0.00]), "H": np.array([4, 4096, 64, 1048576, 256]), "m": 120, "delta": 0.05},
        ]
    if topic == "regularization":
        x = np.linspace(-1.0, 1.0, 25)
        base = 1.0 + 2.0 * x - 1.5 * x ** 2
        return [
            {"name": "D1 lesson 3-row regression", "X": np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]]), "y": np.array([1.0, 2.0, 2.0]), "lambdas": np.array([0.0, 1.0, 10.0, 100.0])},
            {"name": "D2 dense lambda path", "X": np.column_stack([np.ones_like(x), x]), "y": 1.0 + 2.0 * x + 0.15 * np.sin(7.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D3 higher polynomial capacity", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3, x ** 4, x ** 5]), "y": base + 0.10 * np.sin(9.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D4 noisy labels", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3]), "y": base + rng.normal(0.0, 0.18, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D5 unscaled-feature trap", "X": np.column_stack([np.ones_like(x), x, 100.0 * x ** 2]), "y": base + rng.normal(0.0, 0.10, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
        ]
    if topic == "stability":
        sample = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
        long_sample = np.linspace(0.0, 1.0, 50)
        return [
            {"name": "D1 lesson 5-number mean", "sample": sample, "kind": "mean"},
            {"name": "D2 m=50 bounded-range mean", "sample": long_sample, "kind": "mean"},
            {"name": "D3 ridge lambda ladder", "sample": np.column_stack([np.ones(20), np.linspace(-1.0, 1.0, 20)]), "target": 1.0 + np.linspace(-1.0, 1.0, 20), "lambdas": np.array([0.1, 0.3, 1.0, 3.0]), "kind": "ridge"},
            {"name": "D4 compare mean/ridge to 1-NN-style rule", "sample": np.linspace(0.0, 1.0, 20), "kind": "nn_compare"},
            {"name": "D5 outlier removal stress case", "sample": np.array([0.0, 0.1, 0.2, 0.3, 9.0]), "kind": "outlier"},
        ]
    if topic == "nfl":
        return [
            {"name": "D1 one unseen point", "n": 1, "prediction": np.array([0])},
            {"name": "D2 lesson three unseen points", "n": 3, "prediction": np.array([0, 0, 0])},
            {"name": "D3 four unseen points", "n": 4, "prediction": np.array([1, 0, 1, 0])},
            {"name": "D4 biased smooth subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "smooth"},
            {"name": "D5 adversarial mirror subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "mirror"},
        ]
    if topic == "uniform":
        return [
            {"name": "D1 one fixed hypothesis", "H": 1, "m": 500, "epsilon": 0.1},
            {"name": "D2 hundred-hypothesis lesson class", "H": 100, "m": 500, "epsilon": 0.1},
            {"name": "D3 low-m vacuous rungs", "H": 100, "ms": np.array([100, 150, 230, 300]), "epsilon": 0.1},
            {"name": "D4 solve m for delta", "H": 100, "delta": 0.05, "epsilon": 0.1},
            {"name": "D5 correlated non-iid violation", "H": 100, "m": 500, "epsilon": 0.1, "rho": 0.9},
        ]
    raise ValueError(topic)


def all_binary_targets(n):
    rows = list(itertools.product([0, 1], repeat=n))
    return np.array(rows, dtype=int)


def ridge_weights(X, y, lam, penalize_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    penalty = np.eye(X.shape[1])
    if not penalize_intercept:
        penalty[0, 0] = 0.0
    system = X.T @ X + lam * penalty
    rhs = X.T @ y
    weights = np.linalg.solve(system, rhs)
    return weights


def print_rows(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, item in enumerate(row):
            widths[i] = max(widths[i], len(str(item)))
    header = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    print(header)
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(item).ljust(widths[i]) for i, item in enumerate(row)))


## The concept, built once (D1)

For unseen points with every labeling equally likely, compute the average off-training error of a fixed prediction:
$$\frac{1}{2^n}\sum_{f\in\{0,1\}^n}\frac{1}{n}\sum_{i=1}^n\mathbf{1}\{\hat y_i\ne f_i\}.$$
For three unseen points there are $2^3=8$ target vectors, and both $[0,0,0]$ and $[1,0,1]$ average to $0.500$.

In [ ]:

def average_off_training_error(prediction, all_targets):
    prediction = np.asarray(prediction, dtype=int)
    targets = np.asarray(all_targets, dtype=int)
    errors = np.mean(prediction[None, :] != targets, axis=1)
    return float(np.mean(errors)), errors


The exact lesson check enumerates all $8$ targets and proves that two very different learners tie at chance.

In [ ]:

lesson_targets = all_binary_targets(3)
zero_error, zero_errors = average_off_training_error(np.array([0, 0, 0]), lesson_targets)
alt_error, alt_errors = average_off_training_error(np.array([1, 0, 1]), lesson_targets)
print(f"Targets: {len(lesson_targets)}")
print(f"[0, 0, 0] average error: {zero_error:.3f}")
print(f"[1, 0, 1] average error: {alt_error:.3f}")
assert len(lesson_targets) == 8
assert np.isclose(zero_error, 0.500)
assert np.isclose(alt_error, 0.500)


## The dataset ladder

D1 has one unseen point, D2 is the lesson universe of three points, D3 uses four points, D4 restricts to a smooth structured subset, and D5 mirrors that subset adversarially.

In [ ]:

nfl_ladder = theory_ladder("nfl")
rows = []
for dataset in nfl_ladder:
    targets = all_binary_targets(dataset["n"])
    rows.append((dataset["name"], dataset["n"], len(targets), dataset["prediction"].tolist()))
print_rows(rows, ["dataset", "unseen n", "target count", "prediction"])


In [ ]:

def structured_targets(n, kind):
    targets = all_binary_targets(n)
    if kind == "smooth":
        mask = []
        for row in targets:
            changes = int(np.sum(row[1:] != row[:-1]))
            mask.append(changes <= 1)
        return targets[np.array(mask)]
    if kind == "mirror":
        smooth = structured_targets(n, "smooth")
        return 1 - smooth
    return targets

nfl_results = []
for dataset in nfl_ladder:
    kind = dataset.get("subset", "uniform")
    targets = structured_targets(dataset["n"], kind)
    avg_error, errors = average_off_training_error(dataset["prediction"], targets)
    nfl_results.append({"dataset": dataset, "targets": targets, "errors": errors, "avg_error": avg_error})
    print(f"{dataset['name']}: {len(targets)} targets, average off-training error {avg_error:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for j, result in enumerate(nfl_results):
    targets = result["targets"]
    axes[0, j].imshow(targets, aspect="auto", cmap="Greys", vmin=0, vmax=1)
    axes[0, j].set_title(result["dataset"]["name"].split()[0])
    axes[0, j].set_xlabel("unseen point")
    axes[0, j].set_ylabel("target")
    axes[1, j].bar(np.arange(len(result["errors"])), result["errors"])
    axes[1, j].axhline(result["avg_error"], color="red", linestyle="--")
    axes[1, j].set_ylim(0.0, 1.0)
    axes[1, j].set_ylabel("off-training error")
fig.suptitle("No-Free-Lunch target panels and average-error-vs-bias curves")
plt.tight_layout()


## Pitfall on D5: claiming all models are equal on a real task

The equality holds under the uniform prior over all target functions. On structured subsets, a smoothness-biased learner can win; on the mirrored subset it loses by the matching amount. The fix is to state the task assumptions explicitly.

In [ ]:

smooth_prediction = np.array([0, 0, 1, 1])
rough_prediction = np.array([0, 1, 0, 1])
smooth_targets = structured_targets(4, "smooth")
mirror_targets = structured_targets(4, "mirror")
smooth_win, _ = average_off_training_error(smooth_prediction, smooth_targets)
rough_on_smooth, _ = average_off_training_error(rough_prediction, smooth_targets)
smooth_on_mirror, _ = average_off_training_error(smooth_prediction, mirror_targets)
rough_on_mirror, _ = average_off_training_error(rough_prediction, mirror_targets)
print(f"Smooth subset: smooth-biased {smooth_win:.3f}, rough rule {rough_on_smooth:.3f}")
print(f"Mirror subset: smooth-biased {smooth_on_mirror:.3f}, rough rule {rough_on_mirror:.3f}")
print("The theorem averages these advantages and refunds across target families.")


## Evaluate it + practice

- Metric: average off-training error across a target-function family.
- No-skill baseline: chance error 0.5 under the uniform ensemble.
- Cheap sanity check: every fixed prediction averages to 0.5 when all labelings are included.
- Ablation: restrict to a structured subset and watch the tie break.
- Failure signals: using the theorem to dismiss model differences on a non-uniform real domain.

### Practice prompts

- Enumerate five unseen points and verify the uniform 0.5 average.

- Define a sparse-label subset and test two biased predictions.

- Pair each target with its mirror and show their errors sum to one.
